# Fairness Analysis — Catalog Coverage and Popularity Bias

Compares baseline SVD against popularity-penalized SVD on catalog coverage, long-tail coverage, and mean popularity rank.

> Run `02_models.ipynb` first.

In [ ]:
import numpy as np
import pandas as pd

# derive tail film IDs using the same 80/20 cumulative split as the EDA
fp = film_popularity.sort_values('rating_count', ascending=False).reset_index(drop=True)
fp['cumulative_pct'] = fp['rating_count'].cumsum() / fp['rating_count'].sum() * 100
cutoff = (fp['cumulative_pct'] >= 80).idxmax()
tail_movie_ids = set(fp.loc[cutoff + 1:, 'movieId'])

print(f'Total films:      {len(all_movie_ids):,}')
print(f'Tail films:       {len(tail_movie_ids):,}')
print(f'Head films:       {len(all_movie_ids) - len(tail_movie_ids):,}')

In [ ]:
def catalog_coverage(recs, all_movie_ids):
    """Percentage of all films that appear in at least one recommendation list."""
    recommended = {mid for recs_list in recs.values() for mid in recs_list}
    return len(recommended) / len(all_movie_ids) * 100

In [ ]:
def longtail_coverage(recs, tail_movie_ids):
    """Percentage of tail films that appear in at least one recommendation list."""
    recommended = {mid for recs_list in recs.values() for mid in recs_list}
    return len(recommended & tail_movie_ids) / len(tail_movie_ids) * 100

In [ ]:
def mean_popularity_rank(recs, pop_dict):
    """Average log-normalized popularity score across all recommended items (lower = more diverse)."""
    scores = [pop_dict.get(mid, 0.0) for recs_list in recs.values() for mid in recs_list]
    return float(np.mean(scores))

In [ ]:
results = pd.DataFrame({
    'Metric': ['Catalog Coverage (%)', 'Long-Tail Coverage (%)', 'Mean Popularity Rank'],
    'SVD Baseline': [
        round(catalog_coverage(user_recs, all_movie_ids), 4),
        round(longtail_coverage(user_recs, tail_movie_ids), 4),
        round(mean_popularity_rank(user_recs, pop_dict), 4),
    ],
    'Penalized (λ=0.3)': [
        round(catalog_coverage(penalized_recs, all_movie_ids), 4),
        round(longtail_coverage(penalized_recs, tail_movie_ids), 4),
        round(mean_popularity_rank(penalized_recs, pop_dict), 4),
    ],
})

results['Delta'] = (results['Penalized (λ=0.3)'] - results['SVD Baseline']).round(4)

print(results.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

lambda_values = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
rows = []

for lam in lambda_values:
    recs = {}
    for user_id in sample_users:
        recs[user_id] = get_penalized_recommendations(
            user_id, svd, all_movie_ids, user_rated[user_id], lambda_penalty=lam
        )
    rows.append({
        'lambda': lam,
        'catalog_coverage': round(catalog_coverage(recs, all_movie_ids), 4),
        'longtail_coverage': round(longtail_coverage(recs, tail_movie_ids), 4),
    })
    print(f'λ={lam:.1f}  catalog={rows[-1]["catalog_coverage"]:.4f}%  longtail={rows[-1]["longtail_coverage"]:.4f}%')

tradeoff_df = pd.DataFrame(rows)
print()
print(tradeoff_df.to_string(index=False))

# plot
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(tradeoff_df['lambda'], tradeoff_df['catalog_coverage'],
        marker='o', label='Catalog Coverage (%)', color='steelblue')
ax.plot(tradeoff_df['lambda'], tradeoff_df['longtail_coverage'],
        marker='s', label='Long-Tail Coverage (%)', color='darkorange')

for _, row in tradeoff_df.iterrows():
    ax.annotate(f'{row["catalog_coverage"]:.2f}',
                (row['lambda'], row['catalog_coverage']),
                textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8, color='steelblue')
    ax.annotate(f'{row["longtail_coverage"]:.2f}',
                (row['lambda'], row['longtail_coverage']),
                textcoords='offset points', xytext=(0, -14), ha='center', fontsize=8, color='darkorange')

ax.set_xlabel('Lambda (Penalty Strength)')
ax.set_ylabel('Coverage (%)')
ax.set_title('Coverage vs. Popularity Penalty Strength')
ax.set_xticks(lambda_values)
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../outputs/tradeoff_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to outputs/tradeoff_curve.png')

### Lambda selection

λ=0.3 is the best tradeoff — coverage keeps rising with higher λ but at λ=0.5 the penalty starts overriding genuine CF signal.

In [ ]:
from scipy import stats

svd_scores = [pop_dict.get(mid, 0.0) for recs_list in user_recs.values() for mid in recs_list]
penalized_scores = [pop_dict.get(mid, 0.0) for recs_list in penalized_recs.values() for mid in recs_list]

u_stat, p_value = stats.mannwhitneyu(svd_scores, penalized_scores, alternative='greater')

significant = p_value < 0.05
print(f'Mann-Whitney U statistic: {u_stat:,.0f}')
print(f'P-value:                  {p_value:.4e}')
print(f'Significant at p < 0.05:  {significant}')
print()
print(f'SVD baseline   — n={len(svd_scores):,}  mean popularity score={sum(svd_scores)/len(svd_scores):.4f}')
print(f'Penalized recs — n={len(penalized_scores):,}  mean popularity score={sum(penalized_scores)/len(penalized_scores):.4f}')

### Mann-Whitney interpretation

Testing whether SVD recommends stochastically more popular items than the penalized model. Mann-Whitney used because popularity scores are skewed. Significant result means the penalty is genuinely shifting recommendations toward less-exposed films.

In [ ]:
import os
os.makedirs('../outputs', exist_ok=True)

# tradeoff_results.csv
# lambda sweep table produced by the coverage-vs-penalty cell above.
tradeoff_df.to_csv('../outputs/tradeoff_results.csv', index=False)

# fairness_comparison.csv
# three-metric comparison table: SVD Baseline vs Penalized (λ=0.3).
results.to_csv('../outputs/fairness_comparison.csv', index=False)

# mann_whitney_results.csv
# single-row summary of the Mann-Whitney U test on recommendation popularity
# score distributions.
pd.DataFrame({
    'u_statistic':              [u_stat],
    'p_value':                  [p_value],
    'significant_at_0.05':      [significant],
    'svd_mean_pop_score':       [sum(svd_scores)       / len(svd_scores)],
    'penalized_mean_pop_score': [sum(penalized_scores) / len(penalized_scores)],
}).to_csv('../outputs/mann_whitney_results.csv', index=False)

print('Saved  →  outputs/tradeoff_results.csv')
print('Saved  →  outputs/fairness_comparison.csv')
print('Saved  →  outputs/mann_whitney_results.csv')